## Import 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline

from diasense.config import (
    CLINICAL_FEATURES,
    CLUSTERING_FEATURES,
    CLINICAL_THRESHOLDS,
    KMEANS_K,
    K_RANGE,
    FIGURES_DIR,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
)
from diasense.data import load_raw_data, validate_columns
from diasense.preprocessing import (
    ColumnSelector,
    build_preprocessing_pipeline,
    inverse_scale,
    save_artifact,
)
from diasense.clustering import evaluate_k, fit_kmeans
from diasense.risk import (
    add_risk_category,
    learn_risk_mapping,
    save_risk_mapping,
)

sns.set_theme(style="whitegrid")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

#  Load

In [ ]:
df_raw = load_raw_data()
validate_columns(df_raw)

X_raw = df_raw[CLINICAL_FEATURES].copy()

# Fit once on the 8 clinical measurements
preprocessing = build_preprocessing_pipeline()
X_prepared = preprocessing.fit_transform(X_raw)

# Select the 3 features used for K-Means
selector = ColumnSelector(CLUSTERING_FEATURES)
X_cluster = selector.fit_transform(X_prepared)

assert X_cluster.shape == (len(df_raw), 3)
assert not X_cluster.isna().any().any()

print("Raw data:", X_raw.shape)
print("Clustering input:", X_cluster.shape)
print("Selected features:", list(X_cluster.columns))

display(X_cluster.head())

Raw data: (768, 8)
Clustering input: (768, 3)
Selected features: ['Glucose', 'BMI', 'DiabetesPedigreeFunction']


,Glucose,BMI,DiabetesPedigreeFunction
0,0.862287,0.165097,0.468492
1,-1.202229,-0.846404,-0.365061
2,2.009241,-1.323254,0.604397
3,-1.071148,-0.629654,-0.920763
4,0.501816,1.537847,3.001955
